### Final Assignment Iván Montalvo

The following code requires to upload the `results.json` file to this file.

##### Installation of unsloth and other required libraries.

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [2]:
!pip install datasets

In [3]:
!pip install transformers==4.51.3

##### Library import

In [4]:
import torch
from datasets import load_dataset

### Attention!
Due to the size of the model, I was unable to mount my google drive and have the `results.json` ready for usage without having to pay. In order to perform the training, please upload it manually from the shared Google drive. Otherwise, it won't be able to perform.

##### Unsloth intialization of the model

In [5]:
from unsloth import FastLanguageModel

max_seq_length = 1024
load_in_4bit = True
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    dtype = dtype,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit
)

NotImplementedError: Unsloth: No NVIDIA GPU found? Unsloth currently only supports GPUs!

##### Setting unsloth fine-tuning targets

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

##### Setting the fine-tunning dataset

In [ ]:
from unsloth import to_sharegpt
dataset = load_dataset('json', data_files='../content/results.json', split='train')


dataset = to_sharegpt(
    dataset,
    merged_prompt="{text}[[\nYour input is:\n{text}]]",
    output_column_name="text",
    conversation_extension=3
)

In [ ]:
from unsloth import standardize_sharegpt
dataset = standardize_sharegpt(dataset)

##### Setting the chat template

In [ ]:
from unsloth import apply_chat_template

chat_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{SYSTEM}<|eot_id|><|start_header_id|>user<|end_header_id|>

{INPUT}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{OUTPUT}<|eot_id|>"""

dataset = apply_chat_template(
    dataset,
    tokenizer=tokenizer,
    chat_template=chat_template,
    default_system_message = "You are a helpful assistant that gives concise and short answers about languages."
)

##### Setting up the trainer and performing the training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 50,
        num_train_epochs = 1,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
trainer_stats = trainer.train()

##### Setting the inference model and the text streamer

In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "You are a helpful assistant that gives concise and short answers about languages."},
    {"role": "user", "content": "Hi, How are you?"}
]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    pad_token_id = tokenizer.eos_token_id,
    return_tensors = "pt") .to("cuda")


from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids, streamer = text_streamer, max_new_tokens = 128, pad_token_id = tokenizer.eos_token_id)

##### Saving the lora model

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")

##### Interactive cell to perform questions.
In order to change the question, change the `target` string.

In [ ]:
target = "¿Que significa huihichua en Muysca?"
messages = [
        {"role": "user", "content": target}
]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids, streamer = text_streamer, max_new_tokens = 128, pad_token_id = tokenizer.eos_token_id)